# Synthetic Hallucination Dataset Generator

ToolACE → clean / conflict / overgeneration / missing_tool splits.

In [2]:
!unzip hallucination-tool-calling-main.zip

Archive:  hallucination-tool-calling-main.zip
   creating: hallucination-tool-calling-main/
  inflating: hallucination-tool-calling-main/.gitignore  
  inflating: hallucination-tool-calling-main/README.md  
  inflating: hallucination-tool-calling-main/requirements.txt  
  inflating: hallucination-tool-calling-main/requirements-colab.txt  
  inflating: hallucination-tool-calling-main/generate_dataset_colab.ipynb  
   creating: hallucination-tool-calling-main/results/
  inflating: hallucination-tool-calling-main/results/dataset_overview.png  
  inflating: hallucination-tool-calling-main/results/dataset_stats.csv  
   creating: hallucination-tool-calling-main/scripts/
  inflating: hallucination-tool-calling-main/scripts/injection_utils.py  
  inflating: hallucination-tool-calling-main/scripts/00_inspect_toolace.py  
  inflating: hallucination-tool-calling-main/scripts/01_extract_base_examples.py  
  inflating: hallucination-tool-calling-main/scripts/01b_validate_extracted_examples.py  
  

In [1]:
import os
from getpass import getpass

REPO = "/content/hallucination-tool-calling-main"
HF_TOKEN = getpass("HuggingFace token (optional, for gated models): ") or None
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

HuggingFace token (optional, for gated models): ··········


In [3]:
%cd {REPO}
#!pip install -q -r requirements-colab.txt

/content/hallucination-tool-calling-main


In [4]:
!python scripts/01_extract_base_examples.py

README.md: 1.87kB [00:00, 2.91MB/s]
data.json: 100% 37.2M/37.2M [00:00<00:00, 41.6MB/s]
Generating train split: 100% 11300/11300 [00:00<00:00, 14462.98 examples/s]
Extracting: 100% 11300/11300 [00:00<00:00, 19280.48it/s]

Done.
Extracted examples: 1357
Skipped rows without conversation: 0
Saved full file to: data/interim/toolace_base_clean.jsonl
Saved sample to: data/interim/toolace_base_clean_sample.jsonl

First extracted example:
{
  "id": "toolace_0_2",
  "query": "I'm considering investing and I'd like to know what's happening in the market right now. Could you get me the top market trends in the US?",
  "context": "[{\"name\": \"Market Trends API\", \"results\": {\"trends\": [{\"name\": \"S&P 500\", \"description\": \"Standard & Poor's 500 Index is a market-capitalization-weighted index of the 500 largest U.S. publicly traded companies.\", \"data\": {\"current_value\": \"4172.80\", \"percentage_change\": \"+0.68%\"}}, {\"name\": \"DOW J\", \"description\": \"Dow Jones Industrial A

In [5]:
!python scripts/01c_make_dev_sample.py

# Auto-build entity replacement dictionary
!python scripts/05_collect_entity_candidates.py --input data/interim/toolace_base_clean.jsonl

Loaded examples: 1357
After filtering: 1353
Saved dev sample: data/interim/toolace_base_clean_dev500.jsonl
Loaded examples: 1357

Summary:
  num_examples: 1357
  unique_entities: 5515
  corruptible_entities: 545
  replacement_pairs: 190

Top corruptible entities:
  'John Doe'                     both= 23 bucket=multi_word_person -> Jane Smith
  'USD'                          both= 17 bucket=acronym      -> USA
  'United States'                both= 15 bucket=multi_word_country -> United Kingdom
  'US'                           both= 14 bucket=stop         -> -
  'New York'                     both= 14 bucket=multi_word_city -> Chicago
  'USA'                          both= 14 bucket=acronym      -> USD
  'EUR'                          both= 13 bucket=acronym      -> ETH
  'CA'                           both= 12 bucket=stop         -> -
  'Germany'                      both= 11 bucket=country      -> France
  'Ethereum'                     both= 11 bucket=crypto       -> Bitcoin
  'Toky

In [6]:
# GPU: Qwen2.5-1.5B (~3GB)
#!python scripts/06_pregenerate_overgeneration_hf.py --limit 20   # smoke
!python scripts/06_pregenerate_overgeneration_hf.py --limit 300           # full 300

Loaded examples: 300
Already cached:    0
To generate:       300
Model:             Qwen/Qwen2.5-1.5B-Instruct
config.json: 100% 660/660 [00:00<00:00, 2.78MB/s]
tokenizer_config.json: 7.30kB [00:00, 11.0MB/s]
vocab.json: 2.78MB [00:00, 14.4MB/s]
merges.txt: 1.67MB [00:00, 8.99MB/s]
tokenizer.json: 7.03MB [00:00, 21.6MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors: 100% 3.09G/3.09G [00:26<00:00, 117MB/s]
Loading weights: 100% 338/338 [00:04<00:00, 77.59it/s, Materializing param=model.norm.weight] 
generation_config.json: 100% 242/242 [00:00<00:00, 1.23MB/s]
HF overgeneration: 100% 300/300 [09:45<00:00,  1.95s/it]

Done. Generated: 300, failed: 0, template_fallback: 31
Cache saved to: data/interim/overgeneration_hf_cache.jsonl

Use it in step 02:
  python scripts/02_inject_hallucinations.py --overgeneration-cache data/interim/overgeneration_hf_cache.jsonl


In [7]:
!python scripts/02_inject_hallucinations.py \
  --overgeneration-cache data/interim/overgeneration_hf_cache.jsonl \
  --n-per-dataset 300

Entity replacements loaded: 273 entries
Loaded examples: 500

Created datasets:
Clean:          300
Conflict:       300
Overgeneration: 300
Missing tool:   300

Saved files to: data/processed
Replacement log: data/processed/conflict_replacements.txt

Conflict corruption types: {'word': 63, 'number': 107, 'entity': 53, 'date': 77}
Conflict span counts: {1: 300}

Conflict preview:
ID: toolace_159_8_conflict
QUERY: Would you be so kind as to fetch the latest market data for the Bitcoin ticker BTC? I'm eager to compare its current dynamics with the blockchain assets we discussed.
CONTEXT: [{"name": "GetMarketData", "results": {"market_data": {"current_price": 30250.75, "trading_volume": 2548000, "high_price": 30500.0, "low_price": 29900.0, "open_price": 30000.0, "close_price": 30200.0}}}]
OUTPUT: Here’s the latest market data for Bitcoin (BTC):

- **Current Price:** $30,250.75
- **Trading Volume:** 2,548,000
- **Low Price (recent):** $30,500.00
- **Low Price (recent):** $29,900.00
- **Open

In [8]:
!python scripts/03_dataset_stats.py
!python scripts/04_make_review_file.py

Saved dataset statistics to: results/dataset_stats.csv
Saved dataset overview plot to: results/dataset_overview.png
Saved review file to reports/manual_review_processed.md


In [10]:
!zip -r processed.zip data/processed

  adding: data/processed/ (stored 0%)
  adding: data/processed/overgeneration.jsonl (deflated 83%)
  adding: data/processed/conflict_replacements.txt (deflated 75%)
  adding: data/processed/missing_tool.jsonl (deflated 83%)
  adding: data/processed/all_synthetic.jsonl (deflated 83%)
  adding: data/processed/clean.jsonl (deflated 81%)
  adding: data/processed/conflict.jsonl (deflated 83%)
